# Surface-Frame Stiffness Analysis — IROS-workshop pitch sweep

How each controller shapes its **applied translational stiffness** on the surface frame
`[along-track, cross-track, normal]` across the method × grasp-angle sweep. Based on
`stiff_surface_analysis.ipynb`, but self-contained on `analysis_utils.py` and the
pitch_sweep data model.

Figures:

1. **Directional stiffness over training (2×2)** — one panel per direction (normal / along /
   cross), one mean+95%CI line per method.
2. **Directional stiffness, aggregated over methods (1×3)** — the same three directions, all
   runs pooled into one line per direction.
3. **k$_n$ vs k$_\parallel$ scatter** — one point per agent, COLOR = method, SHAPE = grasp
   angle (no error bars).
4. **Stiffness oval — normal-axis (z) tilt** (range-stretched): a 4-column row per grasp angle
   (aggregated over methods) and a 4-column row per method (aggregated over grasp angles).

All stiffness values are reduced end-of-training (`REDUCE="last"`). Figures save as SVG under
`runs/{FOLDER}/plots/iros_stiff/`.

## 1. Imports

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt


# analysis_utils.py sits next to this notebook in iros_workshop_analysis/. Make it
# importable whether the kernel starts here, in data_analysis/, or at the repo root.
def _ensure_on_path():
    d = os.path.abspath(os.getcwd())
    while True:
        for cand in (d,
                     os.path.join(d, "iros_workshop_analysis"),
                     os.path.join(d, "data_analysis", "iros_workshop_analysis")):
            if os.path.isfile(os.path.join(cand, "analysis_utils.py")):
                if cand not in sys.path:
                    sys.path.insert(0, cand)
                return
        parent = os.path.dirname(d)
        if parent == d:
            return
        d = parent


_ensure_on_path()
import importlib
import analysis_utils as au   # data loading + styling + plotting (single source of truth)
importlib.reload(au)

## 2. Global parameters

In [ ]:
# --- Data source (wandb). Same project/tag as the other notebooks; the download is cached,
# so running ANY of them fills the shared cache. Set WANDB_PROJECT/WANDB_TAG = None for a local
# runs/ subfolder via LOCAL_FOLDER.
WANDB_PROJECT = "pitch_sweep"
WANDB_TAG     = "pitch_sweep"
WANDB_ENTITY  = "hur"
LOCAL_FOLDER  = ""
FOLDER_NAME   = f"{WANDB_PROJECT}_{WANDB_TAG}" if (WANDB_PROJECT and WANDB_TAG) else LOCAL_FOLDER
FORCE_WANDB_REFRESH = False

# Confidence band: mean +/- CI_Z * SEM across seeds. 1.96 -> ~95% CI.
CI_Z = 1.96
XLABEL = "Env Steps"
XLIM = None                      # (min, max) or None to autoscale

# How each stiffness scalar is reduced per run (scatter + ovals). "last" = end of training;
# "mean_tail" averages the final few points.
REDUCE = "last"
# Stiffness-oval tilt reconstruction and display scale (see analysis_utils.figure_stiffness_ellipses).
TILT_MODE  = "zaxis"             # normal-axis (z) polar tilt off the true surface normal
SCALE_MODE = "range"            # range-stretched so small shape/tilt differences pop

# The three surface directions to chart over training.
STIFF = au.SURFACE_STIFFNESS_TAGS
DIR_SPECS = [
    {"tag": STIFF["normal"],      "ylabel": "stiffness (N/m)", "title": "Normal (into-surface)"},
    {"tag": STIFF["along_track"], "ylabel": "stiffness (N/m)", "title": "Along-track"},
    {"tag": STIFF["cross_track"], "ylabel": "stiffness (N/m)", "title": "Cross-track"},
]

# Output folder: runs/{FOLDER}/plots/iros_stiff/{name}.svg (overwrites in place; no date stamp).
PLOTS_DIR = os.path.join(au.runs_root(), FOLDER_NAME, "plots", "iros_stiff")
STYLE = au.Style(ci_z=CI_Z, xlabel=XLABEL, xlim=XLIM, plots_dir=PLOTS_DIR)

## 3. Load data

In [ ]:
# Download once (cached). include_eval=True keeps the shared cache identical across the three
# notebooks; this notebook uses only the training runs.
if WANDB_PROJECT and WANDB_TAG:
    au.download_wandb_data(WANDB_PROJECT, WANDB_TAG, entity=WANDB_ENTITY, root=au.runs_root(),
                           force=FORCE_WANDB_REFRESH, include_eval=True)
DATA = au.load_data(FOLDER_NAME, au.runs_root())

## 4. Process data

In [ ]:
COL = au.build_collections(DATA)
print("methods:", COL.methods)
print("angles :", COL.angles)

## 5. Directional stiffness over training — 2×2, by method

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# One panel per surface direction (normal / along-track / cross-track); one mean+95%CI line
# per method (pooled over grasp angles). 2x2 layout (the 4th cell is left empty).
fig = au.figure_lines_panels(COL, DIR_SPECS, STYLE, series="method", ncols=2, sharey=True,
                             suptitle="Surface-frame stiffness over training, by method")
au.save_figure(fig, "stiffness_over_training_by_method", STYLE)
plt.show()

## 6. Directional stiffness over training — aggregated over methods (1×3)

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# The same three directions, but every run pooled into a single line per direction
# (aggregated over methods AND grasp angles) — the overall stiffness profile.
fig = au.figure_lines_panels(COL, DIR_SPECS, STYLE, series="pooled", ncols=3, sharey=True,
                             suptitle="Surface-frame stiffness over training (aggregated over methods)")
au.save_figure(fig, "stiffness_over_training_pooled", STYLE)
plt.show()

## 7. k$_n$ vs k$_\parallel$ scatter (per agent)

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# One point per agent: x = k_along-track, y = k_normal, reduced end-of-training (REDUCE).
# COLOR = method, SHAPE = grasp angle (circle/square/triangle/hexagon). No error bars.
fig = au.figure_pareto(COL, STIFF["along_track"], STIFF["normal"], STYLE,
                       mode=REDUCE, selection_metric=None,
                       xlabel=r"$k_\parallel$  (along-track stiffness)",
                       ylabel=r"$k_n$  (normal / into-surface stiffness)",
                       title=r"$k_n$ vs $k_\parallel$ per agent", pareto=False)
au.save_figure(fig, "kn_vs_kpar_scatter", STYLE)
plt.show()

## 8. Stiffness oval — normal-axis (z) tilt

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 4 columns = grasp angles (aggregated over methods). Vertical = surface normal,
# horizontal = along-track; tilt = normal-axis (z) polar tilt off the surface normal;
# range-stretched scale so shape/tilt differences pop.
fig = au.figure_stiffness_ellipses(COL.by_angle, COL.angles, STYLE,
                                   color_fn=STYLE.acolor, name_fn=STYLE.aname,
                                   reduce=REDUCE, tilt_mode=TILT_MODE, scale_mode=SCALE_MODE,
                                   suptitle="Stiffness oval (normal-axis z tilt) — by grasp angle, aggregated over methods")
au.save_figure(fig, "stiffness_oval_by_angle", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 4 columns = methods (aggregated over grasp angles). Same oval construction as above.
fig = au.figure_stiffness_ellipses(COL.by_method, COL.methods, STYLE,
                                   color_fn=STYLE.mcolor, name_fn=STYLE.mname,
                                   reduce=REDUCE, tilt_mode=TILT_MODE, scale_mode=SCALE_MODE,
                                   suptitle="Stiffness oval (normal-axis z tilt) — by method, aggregated over grasp angles")
au.save_figure(fig, "stiffness_oval_by_method", STYLE)
plt.show()